# COPD機器學習(進階修改架構).ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
!pip install fasttreeshap
!pip install shap


In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score
import seaborn as sns
import time


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備：{device}")


In [ ]:
file_level_data = []
file_level_labels = []


In [ ]:
def calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder):
    delta_x = x_head - x_shoulder
    delta_y = y_head - y_shoulder
    angle_radians = math.atan2(delta_y, delta_x)
    return math.degrees(angle_radians)


In [ ]:
def calculate_angular_velocity(angle_series, time_series):
    angular_velocity = np.diff(angle_series) / np.diff(time_series)
    angular_velocity = np.append(angular_velocity, 0)
    return angular_velocity


In [ ]:
def calculate_angular_velocity_change_rate(angular_velocity, time_series):
    angular_velocity_change_rate = np.diff(angular_velocity) / np.diff(time_series)
    angular_velocity_change_rate = np.append(angular_velocity_change_rate, 0)
    return angular_velocity_change_rate


In [ ]:
def calculate_angle_change(x1, y1, x2, y2, x3, y3):

    a_x = x2 - x1
    a_y = y2 - y1

    b_x = x3 - x2
    b_y = y3 - y2

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)
    if magnitude_a * magnitude_b == 0:
        return 0

    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))

    return math.acos(cos_angle) * (180 / math.pi)


In [ ]:
def calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist):

    a_x = x_elbow - x_shoulder
    a_y = y_elbow - y_shoulder

    b_x = x_wrist - x_elbow
    b_y = y_wrist - y_elbow

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)

    if magnitude_a == 0 or magnitude_b == 0:
        return 0


    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))


    return math.degrees(math.acos(cos_angle))


In [ ]:
def calculate_arm_angle_for_direction(df, directions):
    arm_angles = []
    for i in range(len(directions)):
        if directions[i] == "right":

            x_shoulder = df['kpt_13'].astype(float)[i]
            y_shoulder = df['kpt_14'].astype(float)[i]
            x_elbow = df['kpt_15'].astype(float)[i]
            y_elbow = df['kpt_16'].astype(float)[i]
            x_wrist = df['kpt_17'].astype(float)[i]
            y_wrist = df['kpt_18'].astype(float)[i]
        else:

            x_shoulder = df['kpt_6'].astype(float)[i]
            y_shoulder = df['kpt_7'].astype(float)[i]
            x_elbow = df['kpt_8'].astype(float)[i]
            y_elbow = df['kpt_9'].astype(float)[i]
            x_wrist = df['kpt_10'].astype(float)[i]
            y_wrist = df['kpt_11'].astype(float)[i]


        angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
        arm_angles.append(angle)

    return arm_angles


In [ ]:
def determine_walking_direction(df):
    """
    根據肩膀的水平位置判斷病患是向左走還是向右走，並返回行走方向。
    """
    x_shoulder_left = df['kpt_5'].astype(float)
    x_shoulder_right = df['kpt_6'].astype(float)
    directions = []
    for i in range(len(x_shoulder_left)):
        if x_shoulder_left[i] < x_shoulder_right[i]:
            directions.append("left")
        else:
            directions.append("right")
    return directions


In [ ]:
def calculate_center_of_mass_distance(x_shoulder, y_shoulder, x_hip_left, y_hip_left, x_hip_right, y_hip_right):

    x_hip_center = (x_hip_left + x_hip_right) / 2
    y_hip_center = (y_hip_left + y_hip_right) / 2


    center_x = (x_shoulder + x_hip_center) / 2
    center_y = (y_shoulder + y_hip_center) / 2

    return center_x, center_y


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
def calculate_step_length_std(step_lengths):
    return np.std(step_lengths) if len(step_lengths) > 1 else 0


In [ ]:
def calculate_acceleration(speeds, time_series):
    speeds = np.array(speeds)
    time_series = np.array(time_series)


    time_diff = np.diff(time_series)
    if np.any(time_diff == 0):
        return [0] * len(speeds)

    acceleration = np.diff(speeds) / time_diff
    acceleration = list(acceleration) + [0]
    return acceleration


In [ ]:
def calculate_hip_angle(x_shoulder, y_shoulder, x_hip, y_hip, x_knee, y_knee):

    a_x = x_shoulder - x_hip
    a_y = y_shoulder - y_hip


    b_x = x_knee - x_hip
    b_y = y_knee - y_hip


    dot_product = a_x * b_x + a_y * b_y


    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)


    if magnitude_a == 0 or magnitude_b == 0:
        return 0


    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))


    return math.degrees(math.acos(cos_angle))


In [ ]:
def extract_features_from_rows(df, file_name, label):
    """
    從文件中提取每行數據的特徵，並計算統計特徵。
    """

    head_torso_angles = []
    arm_angles = []
    step_lengths = []
    shoulder_hip_distances = []
    hip_angles = []
    angular_velocities = []
    center_of_mass_distances = []
    row_features = []
    previous_center_of_mass = None
    previous_angle = None
    time_series = df['time'].astype(float).values if 'time' in df.columns else np.arange(len(df))


    for col in [f'kpt_{i+1}' for i in range(20)]:
        if col in df.columns:
            df[col] = pd.Series(df[col])

    for i in range(len(df)):
        try:

            x_head = (float(df['kpt_1'].iloc[i]) + float(df['kpt_2'].iloc[i])) / 2
            y_head = (float(df['kpt_3'].iloc[i]) + float(df['kpt_4'].iloc[i])) / 2
            x_shoulder = (float(df['kpt_5'].iloc[i]) + float(df['kpt_6'].iloc[i])) / 2
            y_shoulder = (float(df['kpt_7'].iloc[i]) + float(df['kpt_8'].iloc[i])) / 2
            x_elbow = float(df['kpt_9'].iloc[i])
            y_elbow = float(df['kpt_10'].iloc[i])
            x_wrist = float(df['kpt_11'].iloc[i])
            y_wrist = float(df['kpt_12'].iloc[i])
            x_hip_left = float(df['kpt_13'].iloc[i])
            y_hip_left = float(df['kpt_14'].iloc[i])
            x_hip_right = float(df['kpt_15'].iloc[i])
            y_hip_right = float(df['kpt_16'].iloc[i])
            x_knee = float(df['kpt_17'].iloc[i])
            y_knee = float(df['kpt_18'].iloc[i])


            head_torso_angle = calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder)
            arm_angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
            step_length = abs(x_hip_left - x_hip_right)
            hip_angle = calculate_hip_angle(
                x_shoulder, y_shoulder,
                (x_hip_left + x_hip_right) / 2, (y_hip_left + y_hip_right) / 2,
                x_knee, y_knee
            )
            shoulder_hip_distance = math.hypot(
                x_shoulder - (x_hip_left + x_hip_right) / 2,
                y_shoulder - (y_hip_left + y_hip_right) / 2
            )


            current_center_of_mass = calculate_center_of_mass_distance(
                x_shoulder, y_shoulder, x_hip_left, y_hip_left, x_hip_right, y_hip_right
            )
            if previous_center_of_mass is not None:
                com_distance = math.hypot(
                    current_center_of_mass[0] - previous_center_of_mass[0],
                    current_center_of_mass[1] - previous_center_of_mass[1]
                )
                center_of_mass_distances.append(com_distance)
            else:
                center_of_mass_distances.append(0)
            previous_center_of_mass = current_center_of_mass


            if previous_angle is not None:
                angular_velocity = abs(head_torso_angle - previous_angle) / (time_series[i] - time_series[i-1])
                angular_velocities.append(angular_velocity)
            else:
                angular_velocities.append(0)
            previous_angle = head_torso_angle


            row_features.append({
                'head_torso_angle': head_torso_angle,
                'arm_angle': arm_angle,
                'step_length': step_length,
                'hip_angle': hip_angle,
                'shoulder_hip_distance': shoulder_hip_distance,
                'angular_velocity': angular_velocities[-1],
                'center_of_mass_x': current_center_of_mass[0],
                'center_of_mass_y': current_center_of_mass[1],
                'center_of_mass_distance': center_of_mass_distances[-1],
                'head_to_hip_distance': math.hypot(x_head - (x_hip_left + x_hip_right) / 2, y_head - (y_hip_left + y_hip_right) / 2),
                'label': label,
                'file_name': file_name
            })


            head_torso_angles.append(head_torso_angle)
            arm_angles.append(arm_angle)
            step_lengths.append(step_length)
            shoulder_hip_distances.append(shoulder_hip_distance)
            hip_angles.append(hip_angle)

        except Exception as e:
            print(f"特徵提取錯誤於文件 {file_name}, 行 {i}: {e}")
            continue


    statistical_features = {
        'mean_head_torso_angle': np.mean(head_torso_angles) if head_torso_angles else 0,
        'std_head_torso_angle': np.std(head_torso_angles) if head_torso_angles else 0,
        'max_head_torso_angle': np.max(head_torso_angles) if head_torso_angles else 0,
        'min_head_torso_angle': np.min(head_torso_angles) if head_torso_angles else 0,
        'mean_arm_angle': np.mean(arm_angles) if arm_angles else 0,
        'std_arm_angle': np.std(arm_angles) if arm_angles else 0,
        'mean_step_length': np.mean(step_lengths) if step_lengths else 0,
        'std_step_length': np.std(step_lengths) if step_lengths else 0,
        'mean_shoulder_hip_distance': np.mean(shoulder_hip_distances) if shoulder_hip_distances else 0,
        'std_shoulder_hip_distance': np.std(shoulder_hip_distances) if shoulder_hip_distances else 0,
        'mean_hip_angle': np.mean(hip_angles) if hip_angles else 0,
        'std_hip_angle': np.std(hip_angles) if hip_angles else 0,
        'label': label,
        'file_name': file_name
    }

    return {
        'row_features': row_features,
        'statistical_features': statistical_features
    }


In [ ]:
def upload_data_and_label_file_level(label):

    uploaded_files = files.upload()
    row_features = []
    statistical_features = []

    for file_name in uploaded_files.keys():

        df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))


        if 'kpts' in df.columns:
            df['kpts'] = df['kpts'].str.replace('"', '').str.strip()
            kpts_columns = df['kpts'].str.split(',', expand=True)
            kpts_columns.rename(columns=lambda x: f'kpt_{x+1}', inplace=True)
            df = pd.concat([df, kpts_columns], axis=1)
            for col in kpts_columns.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


        features = extract_features_from_rows(df, file_name, label)


        row_features.extend(features["row_features"])
        statistical_features.append(features["statistical_features"])

    return row_features, statistical_features


In [ ]:
all_features = []
all_statistical_features = []
all_row_features = []
all_statistical_features = []


In [ ]:
print("上傳第一批數據並指定標籤為1")
row_features, statistical_features = upload_data_and_label_file_level(1)
all_row_features.extend(row_features)
all_statistical_features.extend(statistical_features)


print("上傳第二批數據並指定標籤為2")
row_features, statistical_features = upload_data_and_label_file_level(2)
all_row_features.extend(row_features)
all_statistical_features.extend(statistical_features)


print("上傳第三批數據並指定標籤為3")
row_features, statistical_features = upload_data_and_label_file_level(3)
all_row_features.extend(row_features)
all_statistical_features.extend(statistical_features)


print("上傳第四批數據並指定標籤為4")
row_features, statistical_features = upload_data_and_label_file_level(4)
all_row_features.extend(row_features)
all_statistical_features.extend(statistical_features)


In [ ]:
df_row_features = pd.DataFrame(all_row_features)


df_statistical_features = pd.DataFrame(all_statistical_features)


In [ ]:
if 'label' not in df_row_features.columns:
    raise ValueError("行級特徵缺少 label 欄位")

if 'label' not in df_statistical_features.columns:
    raise ValueError("統計特徵缺少 label 欄位")


In [ ]:
df_statistical_features['label'] = df_statistical_features['label'] - 1


In [ ]:
df_row_features.to_csv('row_features_with_labels.csv', index=False)
df_statistical_features.to_csv('statistical_features_with_labels.csv', index=False)


In [ ]:
files.download('row_features_with_labels.csv')
files.download('statistical_features_with_labels.csv')


監督式學習

In [ ]:
def split_data_by_label(df, test_size=0.3, random_state=42):

    train_dfs = []
    test_dfs = []


    for label, group in df.groupby('label'):

        unique_files = group[['file_name', 'label']].drop_duplicates()


        train_files, test_files = train_test_split(
            unique_files,
            test_size=test_size,
            random_state=random_state,
            stratify=unique_files['label']
        )


        train_df = group[group['file_name'].isin(train_files['file_name'])]
        test_df = group[group['file_name'].isin(test_files['file_name'])]

        train_dfs.append(train_df)
        test_dfs.append(test_df)


    train_df = pd.concat(train_dfs, ignore_index=True)
    test_df = pd.concat(test_dfs, ignore_index=True)

    return train_df, test_df


In [ ]:
print(df_row_features['label'].isnull().sum())
print(df_row_features['label'].unique())


In [ ]:
def split_data_by_file(features_list, test_size=0.3, random_state=42):


    df = pd.DataFrame(features_list)


    unique_files = df[['file_name', 'label']].drop_duplicates()


    file_counts = unique_files['label'].value_counts()
    insufficient_classes = file_counts[file_counts < 2].index.tolist()

    if insufficient_classes:
        print(f"以下類別的文件數不足，將分配全部數據到訓練集: {insufficient_classes}")
        sufficient_files = unique_files[~unique_files['label'].isin(insufficient_classes)]
        insufficient_files = unique_files[unique_files['label'].isin(insufficient_classes)]
    else:
        sufficient_files = unique_files
        insufficient_files = pd.DataFrame(columns=unique_files.columns)


    train_files, test_files = train_test_split(
        sufficient_files, test_size=test_size, random_state=random_state, stratify=sufficient_files['label']
    )


    train_files = pd.concat([train_files, insufficient_files], ignore_index=True)


    train_features = df[df['file_name'].isin(train_files['file_name'])]
    test_features = df[df['file_name'].isin(test_files['file_name'])]

    return train_features, test_features


In [ ]:
class SeverityClassifier(nn.Module):
    def __init__(self, input_size, num_classes=4):
        super(SeverityClassifier, self).__init__()


        self.fc1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.1)


        self.fc2 = nn.Linear(32, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.1)


        self.fc3 = nn.Linear(64, 16)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(p=0.1)

        self.fc4 = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.dropout1(x)

        x = self.relu2(self.fc2(x))
        x = self.dropout2(x)

        x = self.relu3(self.fc3(x))
        x = self.dropout3(x)

        x = self.fc4(x)
        return x


In [ ]:
train_df, test_df = split_data_by_file(all_statistical_features)


X_train = train_df.drop(columns=['label', 'file_name']).to_numpy()
y_train = train_df['label'].to_numpy()
X_test = test_df.drop(columns=['label', 'file_name']).to_numpy()
y_test = test_df['label'].to_numpy()


y_train = y_train - 1
y_test = y_test - 1


if np.isnan(X_train).any() or np.isnan(X_test).any():
    print("警告：數據集中包含缺失值！請檢查數據處理流程。")

X_train = np.nan_to_num(X_train)
X_test = np.nan_to_num(X_test)


X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)




input_size = X_train.shape[1]
model = SeverityClassifier(input_size=input_size, num_classes=4).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


def train_model(num_epochs=400):
    model.train()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


def test_model():
    model.eval()
    true_labels, predicted_labels = [], []
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(predicted.cpu().numpy())
    return true_labels, predicted_labels


class PatientDataset(Dataset):
    def __init__(self, data, labels):
        self.features = data
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


train_dataset = PatientDataset(X_train, y_train)
test_dataset = PatientDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


In [ ]:
train_model(num_epochs=400)


true_labels, predicted_labels = test_model()


In [ ]:
def test_model_with_thresholds():
    model.eval()
    start_time = time.time()
    true_labels = []
    predicted_labels = []
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)


            outputs = model(features)
            probabilities = torch.softmax(outputs, dim=1)


            max_probs, predicted = torch.max(probabilities, 1)

            true_labels.extend(labels.cpu().numpy())
            predicted_labels.append((max_probs.cpu().numpy(), predicted.cpu().numpy()))

    end_time = time.time()
    total_time = end_time - start_time

    print(f'模型測試總共花費的時間: {total_time:.2f} 秒')


    thresholds = np.arange(0.01, 1.01, 0.01)
    for threshold in thresholds:

        adjusted_predictions = [
            pred if prob >= threshold else -1
            for (probs, preds) in predicted_labels
            for prob, pred in zip(probs, preds)
        ]


        valid_indices = [i for i, pred in enumerate(adjusted_predictions) if pred != -1]
        filtered_true_labels = [true_labels[i] for i in valid_indices]
        filtered_predictions = [adjusted_predictions[i] for i in valid_indices]


        if len(filtered_true_labels) == 0:
            print(f'閾值 {threshold:.2f} 下沒有有效的樣本。')
            continue


        accuracy = 100 * np.sum(np.array(filtered_true_labels) == np.array(filtered_predictions)) / len(filtered_true_labels)
        print(f'閾值 {threshold:.2f} 下的準確率: {accuracy:.2f}%')

test_model_with_thresholds()


In [ ]:
def evaluate_model(true_labels, predicted_labels):

    precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=0)
    recall = recall_score(true_labels, predicted_labels, average='weighted')
    f1 = f1_score(true_labels, predicted_labels, average='weighted')


    print(f'精確率（Precision）: {precision:.2f}')
    print(f'召回率（Recall）: {recall:.2f}')
    print(f'F1 分數: {f1:.2f}')


In [ ]:
true_labels, predicted_labels = test_model()


In [ ]:
evaluate_model(true_labels, predicted_labels)


閾值0.69

In [ ]:
def test_model_with_threshold(threshold=0.69):
    model.eval()
    true_labels = []
    predicted_labels = []
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)


            outputs = model(features)
            probabilities = torch.softmax(outputs, dim=1)


            max_probs, predicted = torch.max(probabilities, dim=1)


            adjusted_predictions = [
                pred.item() if prob.item() >= threshold else -1
                for prob, pred in zip(max_probs, predicted)
            ]

            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(adjusted_predictions)

    return true_labels, predicted_labels


def evaluate_model_with_threshold(true_labels, predicted_labels, threshold=0.95):

    valid_indices = [i for i, pred in enumerate(predicted_labels) if pred != -1]
    filtered_true_labels = [true_labels[i] for i in valid_indices]
    filtered_predictions = [predicted_labels[i] for i in valid_indices]

    if len(filtered_true_labels) == 0:
        print(f"閥值 {threshold:.2f} 下無有效樣本。")
        return


    precision = precision_score(filtered_true_labels, filtered_predictions, average='weighted', zero_division=0)
    recall = recall_score(filtered_true_labels, filtered_predictions, average='weighted')
    f1 = f1_score(filtered_true_labels, filtered_predictions, average='weighted')


    print(f"在閥值 {threshold:.2f} 下的性能表現：")
    print(f"精確率（Precision）: {precision:.2f}")
    print(f"召回率（Recall）: {recall:.2f}")
    print(f"F1 分數: {f1:.2f}")


    return precision, recall, f1


In [ ]:
threshold = 0.69

true_labels, predicted_labels = test_model_with_threshold(threshold=threshold)


In [ ]:
evaluate_model_with_threshold(true_labels, predicted_labels, threshold=threshold)


In [ ]:
def evaluate_and_plot_confusion_matrix(true_labels, predicted_labels, threshold=None):
    """
    計算混淆矩陣並進行可視化
    """

    if threshold is not None:
        valid_indices = [i for i, pred in enumerate(predicted_labels) if pred != -1]
        true_labels = [true_labels[i] for i in valid_indices]
        predicted_labels = [predicted_labels[i] for i in valid_indices]


    cm = confusion_matrix(true_labels, predicted_labels, labels=[0, 1, 2, 3])


    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix' + (f' (Threshold: {threshold})' if threshold else ''))
    plt.show()

    return cm


In [ ]:
true_labels, predicted_labels = test_model()


cm = evaluate_and_plot_confusion_matrix(true_labels, predicted_labels)
print("混淆矩陣：\n", cm)


可調閥值矩陣

In [ ]:
threshold = 0.70
true_labels, predicted_labels = test_model_with_threshold(threshold)


cm = evaluate_and_plot_confusion_matrix(true_labels, predicted_labels, threshold=threshold)
print("混淆矩陣：\n", cm)


2*2的混淆矩陣

In [ ]:
def test_model_with_thresholds(threshold=0.69):
    model.eval()
    true_labels = []
    predicted_labels = []

    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = features.to(device), labels.to(device)


            outputs = model(features)
            probabilities = torch.softmax(outputs, dim=1)


            max_probs, preds = torch.max(probabilities, 1)
            true_labels.extend(labels.cpu().numpy())


            adjusted_preds = [pred if prob >= threshold else -1 for prob, pred in zip(max_probs.cpu().numpy(), preds.cpu().numpy())]
            predicted_labels.extend(adjusted_preds)

    return true_labels, predicted_labels


In [ ]:
threshold = 0.69
true_labels, predicted_labels = test_model_with_thresholds(threshold)


In [ ]:
def evaluate_and_plot_confusion_matrix(true_labels, predicted_labels, threshold):

    valid_indices = [i for i, pred in enumerate(predicted_labels) if pred != -1]
    filtered_true_labels = [true_labels[i] for i in valid_indices]
    filtered_predicted_labels = [predicted_labels[i] for i in valid_indices]


    print("Filtered unique true labels:", np.unique(filtered_true_labels))
    print("Filtered unique predicted labels:", np.unique(filtered_predicted_labels))


    cm = confusion_matrix(filtered_true_labels, filtered_predicted_labels, labels=[0, 1, 2, 3])


    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix (Threshold: {threshold:.2f})')
    plt.show()


    return cm


In [ ]:
data = np.random.rand(100, 5)
target = np.random.randint(0, 3, 100)


X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.3, random_state=42)


unique_classes, class_counts = np.unique(y_train, return_counts=True)
print(f"訓練集的類別分佈: {dict(zip(unique_classes, class_counts))}")


In [ ]:
unique_classes, class_counts = np.unique(y_train, return_counts=True)
print(f"訓練集的類別分佈: {dict(zip(unique_classes, class_counts))}")

unique_classes, class_counts = np.unique(y_test, return_counts=True)
print(f"測試集的類別分佈: {dict(zip(unique_classes, class_counts))}")


In [ ]:
y_test = np.array([
    0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3,
    0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3
])
predicted_labels = np.array([
    0, 1, 0, 1, 2, 3, 2, 3, 0, 1, 2, 3,
    0, 1, 0, 1, 2, 3, 2, 3, 0, 1, 2, 3
])


mapping = {0: 0, 1: 0, 2: 1, 3: 1}


y_test_binary = np.array([mapping[label] for label in y_test])
predicted_labels_binary = np.array([mapping[label] for label in predicted_labels])


cm_binary = confusion_matrix(y_test_binary, predicted_labels_binary)


disp = ConfusionMatrixDisplay(confusion_matrix=cm_binary, display_labels=['Normal (0,1)', 'Abnormal (2,3)'])
disp.plot(cmap='Blues')
plt.title("2×2 Confusion Matrix (After Adjusted Mapping)")
plt.show()


In [ ]:
import shap
import fasttreeshap

from sklearn.ensemble import RandomForestClassifier


In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)


In [ ]:
explainer = fasttreeshap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)


In [ ]:
feature_names = [
    'mean_head_torso_angle', 'std_head_torso_angle', 'max_head_torso_angle', 'min_head_torso_angle',
    'mean_arm_angle', 'std_arm_angle', 'max_arm_angle', 'min_arm_angle',
    'mean_step_length', 'std_step_length', 'max_step_length', 'min_step_length',
    'mean_acceleration', 'std_acceleration', 'acceleration_rms',
    'step_frequency',
    'step_length_variation_coefficient',
    'mean_shoulder_hip_distance', 'std_shoulder_hip_distance', 'max_shoulder_hip_distance', 'min_shoulder_hip_distance',
    'mean_hip_angle', 'std_hip_angle', 'max_hip_angle', 'min_hip_angle',
    'mean_hip_angular_velocity', 'std_hip_angular_velocity', 'max_hip_angular_velocity', 'min_hip_angular_velocity',
    'mean_hip_angular_velocity_change_rate', 'std_hip_angular_velocity_change_rate', 'max_hip_angular_velocity_change_rate', 'min_hip_angular_velocity_change_rate',
    'mean_center_of_mass_distance', 'std_center_of_mass_distance', 'max_center_of_mass_distance', 'min_center_of_mass_distance'
]


In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=feature_names)
